# Distillation

Trains the deployable EfficientNet-B0 student against the teacher's full output
distribution rather than only the hard label.

The teacher's relative confidence across *wrong* classes is the signal a one-hot target
throws away: that a tiramisu photo looks somewhat like chocolate cake and nothing at all
like a caesar salad. The student is the model that actually ships, so this is where the
accuracy has to end up.

## Prerequisites

The teacher checkpoint from notebook 03, uploaded as a private Kaggle dataset and attached
to this notebook. Retraining the teacher here instead would cost 6 to 8 GPU hours per
distillation experiment.

Set `TEACHER` below to the attached path, usually `/kaggle/input/<dataset-slug>/best.pt`.

**Before running:** phone-verified account, Accelerator GPU T4 x2, Internet on.

## Cost

Distillation adds one teacher forward pass per batch. No backward pass, so expect roughly
20 to 40 percent over a plain student run rather than double. Budget 3 to 5 hours.

Caching the teacher's logits once would remove even that, and it does not work here: the
cache would hold the teacher's opinion of a crop the student never sees, because
augmentation redraws every epoch. The targets would be wrong in a way the loss curve does
not reveal.

In [ ]:
!git clone --depth 1 https://github.com/simonkundrik/plate-vision.git /kaggle/working/plate-vision
%pip install -q -e "/kaggle/working/plate-vision/model[train,export]"

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 before running.")

In [ ]:
%cd /kaggle/working/plate-vision/model
!python data/download_food101.py --out /kaggle/temp/food101

## Check the teacher before spending hours on it

The checkpoint records its own backbone and class count, so this confirms it loads and
predicts the right width. A mismatch caught here costs seconds; caught by the training
script it costs a queue slot.

In [ ]:
from pathlib import Path

from platevision import checkpoint

TEACHER = Path("/kaggle/input/plate-vision-teacher/best.pt")

teacher, payload = checkpoint.restore_classifier(TEACHER)
print("backbone:  ", payload["backbone"])
print("classes:   ", payload["num_classes"])
print("teacher acc", round(payload["best_metric"], 2))

## Distil

`--kd-alpha 0.7` weights the soft loss more heavily than the hard labels, which is the usual
starting point when the teacher is clearly stronger than the student. `--kd-temperature 4.0`
softens both distributions enough that the teacher's ranking of wrong classes carries
signal.

The soft loss is scaled by temperature squared, so alpha keeps meaning the same mix as the
temperature changes. Without that scaling, raising the temperature would quietly shrink the
soft term and tuning one knob would move the other.

In [ ]:
!python scripts/train_classifier.py \
    --data-root /kaggle/temp/food101/food-101 \
    --out /kaggle/working/runs/distilled \
    --backbone efficientnet_b0 \
    --teacher {TEACHER} --kd-alpha 0.7 --kd-temperature 4.0 \
    --epochs 30 --batch-size 128 --lr 1e-3 --weight-decay 0.05 \
    --label-smoothing 0.1 --mixup-alpha 0.2 --cutmix-alpha 1.0 --ema \
    --workers 4 --amp

## Did it help

The comparison that matters is the distilled student against the *undistilled student on the
same recipe*, not against the baseline from notebook 01. The baseline had no mixup, no EMA,
and no label smoothing, so comparing against it would credit distillation with everything
the recipe contributed.

In [ ]:
import json


def best_of(path, split="val"):
    history = json.loads(Path(path).read_text())
    scores = [e["top1"] for e in history if e["split"] == split]
    return max(scores) if scores else float("nan")


runs = {
    "student, recipe only": "/kaggle/working/runs/ablation/plus_ema/history.json",
    "student, distilled": "/kaggle/working/runs/distilled/history.json",
}

for name, path in runs.items():
    if Path(path).exists():
        live = best_of(path, "val")
        ema = best_of(path, "ema")
        print(f"{name:<22} live {live:6.2f}%   ema {ema:6.2f}%")
    else:
        print(f"{name:<22} not found: {path}")

print(f"\nteacher                {payload['best_metric']:6.2f}%")

## Reading it

The student should land between its undistilled self and the teacher. Landing at or above
the teacher on a 101-class problem with this size gap would be surprising, and is more
likely a sign that the two runs are not comparable than a genuine result: check that the
epoch count, recipe flags, and evaluation split actually match before believing it.

If distillation gains nothing, the likely causes in order are: the teacher is not actually
much better than the student, alpha is too low for the gap, or the temperature is too low
for the teacher's confidence to carry ranking information.